In [ ]:
# SaaS Customer Churn Lab — Rewritten Solution
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="ticks")
plt.rcParams["figure.figsize"] = (9, 5)

np.random.seed(42)
N = 1200

tenure = np.random.exponential(scale=18, size=N).clip(1, 72).astype(int)
contract = np.random.choice(
    ["Month-to-Month", "One-Year", "Two-Year"], size=N,
    p=[0.55, 0.25, 0.20]
)
support_tickets = np.random.poisson(lam=1.8, size=N) + (
    contract == "Month-to-Month"
) * np.random.poisson(lam=1.2, size=N)
monthly_charges = np.random.normal(75.0, 25.0, size=N).clip(20.0, 150.0)
tech_support = np.random.choice(
    ["Yes", "No", "No internet"], size=N,
    p=[0.40, 0.45, 0.15]
)
payment_method = np.random.choice(
    ["Electronic Check", "Bank Transfer", "Credit Card"], size=N,
    p=[0.40, 0.30, 0.30]
)

churn_logits = (
    -2.0 + 0.03 * monthly_charges - 0.05 * tenure
    + 0.45 * support_tickets
    + 1.2 * (contract == "Month-to-Month")
    - 1.0 * (contract == "Two-Year")
    + 0.8 * (payment_method == "Electronic Check")
)
churn_prob = 1 / (1 + np.exp(-churn_logits))
churn = np.where(np.random.rand(N) < churn_prob, "Yes", "No")

total_charges = tenure * monthly_charges + np.random.normal(0, 50, size=N)
total_charges[np.random.choice(N, size=15, replace=False)] = np.nan

df = pd.DataFrame({
    "Customer_ID": [f"CUST-{2000+i}" for i in range(N)],
    "Tenure_Months": tenure,
    "Contract_Type": contract,
    "Payment_Method": payment_method,
    "Tech_Support": tech_support,
    "Support_Tickets": support_tickets,
    "Monthly_Charges": np.round(monthly_charges, 2),
    "Total_Charges": np.round(total_charges, 2),
    "Churn": churn
})
df.to_csv("saas_churn_data.csv", index=False)
print("Dataset created:", df.shape)


In [ ]:
# TASK 1.1 — Tenure vs Churn
plt.figure(figsize=(8, 4))
sns.violinplot(
    data=df, x="Churn", y="Tenure_Months",
    inner="quartile", cut=0
)
plt.title("Tenure Distribution by Churn Status")
plt.xlabel("Churn")
plt.ylabel("Tenure (months)")
plt.tight_layout()
plt.show()

tenure_summary = df.groupby("Churn")["Tenure_Months"].agg(
    Median="median",
    Q1=lambda x: x.quantile(0.25),
    Q3=lambda x: x.quantile(0.75)
)
tenure_summary["IQR"] = tenure_summary["Q3"] - tenure_summary["Q1"]
display(tenure_summary.round(2))

# Answer:
# The churned group has a lower median tenure than the retained group.
# This indicates that retention efforts should begin early in the customer life cycle.


In [ ]:
# TASK 1.2 — Contract Type and Technical Support
segment_rates = pd.crosstab(
    [df["Contract_Type"], df["Tech_Support"]],
    df["Churn"],
    normalize="index"
).mul(100)

print("Churn percentage within each segment:")
display(segment_rates.round(2))

segment_rates.plot(kind="barh", figsize=(10, 6))
plt.title("Churn Rate by Contract and Technical Support")
plt.xlabel("Percentage")
plt.ylabel("Customer segment")
plt.tight_layout()
plt.show()

# Answer:
# Contract duration provides a clear separation in churn behavior. The
# month-to-month groups are the most exposed, while longer contracts show
# substantially more retention.


In [ ]:
# TASK 1.3 — Monthly Charges vs Total Charges
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df, x="Monthly_Charges", y="Total_Charges",
    style="Churn", alpha=0.65
)
plt.title("Monthly Charges vs Total Charges")
plt.xlabel("Monthly charges")
plt.ylabel("Total charges")
plt.tight_layout()
plt.show()

charge_pairs = df[["Monthly_Charges", "Total_Charges"]].dropna()
pearson_corr = charge_pairs["Monthly_Charges"].corr(
    charge_pairs["Total_Charges"], method="pearson"
)
spearman_corr = charge_pairs["Monthly_Charges"].corr(
    charge_pairs["Total_Charges"], method="spearman"
)
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"Spearman correlation: {spearman_corr:.3f}")

# Answer:
# Both measures show a positive association. Total charges reflect both the
# recurring monthly amount and how long the customer has remained active.


In [ ]:
# TASK 2.1 — Correlation Heatmap
analysis_df = df.copy()
analysis_df["Churn_Flag"] = analysis_df["Churn"].eq("Yes").astype(int)

corr_matrix = analysis_df.select_dtypes(include="number").corr()

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr_matrix, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0
)
plt.title("Correlation Matrix of Numeric Variables")
plt.tight_layout()
plt.show()

print("Correlations with Churn_Flag:")
display(corr_matrix["Churn_Flag"].drop("Churn_Flag").sort_values())


In [ ]:
# TASK 2.2 — Multidimensional Segmentation
grid = sns.FacetGrid(
    df, col="Contract_Type", hue="Churn",
    height=4.2, sharex=True, sharey=True
)
grid.map_dataframe(
    sns.scatterplot,
    x="Support_Tickets", y="Monthly_Charges", alpha=0.65
)
grid.add_legend()
grid.set_axis_labels("Support tickets", "Monthly charges")
grid.set_titles("Contract: {col_name}")
plt.show()

# Answer:
# Separating the plot by contract type makes segment differences easier to
# observe. Support burden and pricing do not behave identically across all
# contract groups.


In [ ]:
# TASK 3.1 — Missing Total Charges
missing_before = int(df["Total_Charges"].isna().sum())

expected_total = df["Tenure_Months"] * df["Monthly_Charges"]
df.loc[df["Total_Charges"].isna(), "Total_Charges"] = expected_total[
    df["Total_Charges"].isna()
]

print("Missing before imputation:", missing_before)
print("Missing after imputation:", int(df["Total_Charges"].isna().sum()))

# Answer:
# Customer-specific calculation is preferable to a mean/median because the
# assignment provides a direct relationship between tenure, monthly charges,
# and expected accumulated charges.


In [ ]:
# TASK 3.2 — Support Ticket Outliers
q1_tickets = df["Support_Tickets"].quantile(0.25)
q3_tickets = df["Support_Tickets"].quantile(0.75)
ticket_iqr = q3_tickets - q1_tickets
upper_fence = q3_tickets + 1.5 * ticket_iqr

ticket_outliers = df[df["Support_Tickets"] > upper_fence]
outlier_rate = ticket_outliers["Churn"].eq("Yes").mean() * 100

print(f"Q1: {q1_tickets:.2f}")
print(f"Q3: {q3_tickets:.2f}")
print(f"IQR: {ticket_iqr:.2f}")
print(f"Upper fence: {upper_fence:.2f}")
print(f"Outlier accounts: {len(ticket_outliers)}")
print(f"Churn rate among outliers: {outlier_rate:.2f}%")

# Answer:
# These unusually high-ticket accounts are a useful group for proactive
# service review, although an outlier label alone does not prove churn.


In [ ]:
# TASK 4 — Feature Engineering
df["Ticket_Velocity"] = df["Support_Tickets"] / (df["Tenure_Months"] + 1)

# Alternative second feature: high recurring-charge exposure.
charge_median = df["Monthly_Charges"].median()
df["High_Charge_Flag"] = (
    df["Monthly_Charges"] >= charge_median
).astype(int)

# Retain the assignment's risk flag as a third useful feature.
df["High_Risk_Flag"] = (
    (df["Contract_Type"] == "Month-to-Month") &
    (df["Tech_Support"] == "No")
).astype(int)

print("Churn distribution by High_Charge_Flag:")
display(
    df.groupby("High_Charge_Flag")["Churn"]
      .value_counts(normalize=True)
      .unstack()
      .mul(100).round(2)
)

print("Ticket velocity by churn:")
display(
    df.groupby("Churn")["Ticket_Velocity"]
      .agg(["count", "median", "mean", "std"])
      .round(3)
)

# Answer:
# Ticket velocity measures support activity relative to tenure, while the
# high-charge flag provides a simple financial segmentation.


# TASK 6 — Strategic Insights

### Insight 1 — Contract type is a strong retention lever
- **Claim:** Customers with flexible month-to-month agreements are the clearest high-risk group.
- **Evidence:** The normalized contract/support table shows much higher churn percentages for month-to-month segments than for longer contracts.
- **Business Action:** Create a contract-conversion campaign with temporary price incentives or added benefits.

### Insight 2 — Support-heavy customers need service recovery
- **Claim:** Unusually high support-ticket volume is a practical early warning signal.
- **Evidence:** The IQR rule identifies a specific population of high-ticket accounts whose churn rate can be compared with the full customer base.
- **Business Action:** Escalate repeated-ticket customers to priority support and monitor resolution quality.

### Insight 3 — Early lifecycle retention matters
- **Claim:** Churned customers tend to have shorter tenure.
- **Evidence:** The tenure summary shows a lower median tenure for churned customers than for retained customers.
- **Business Action:** Add onboarding reviews and proactive check-ins during the first few months.

### Insight 4 — Total spend should not be read independently of tenure
- **Claim:** Monthly charges and total charges move together, but accumulated spend also reflects time with the service.
- **Evidence:** Pearson and Spearman correlations are positive, while the scatterplot shows an upward relationship.
- **Business Action:** Evaluate pricing offers using tenure and lifetime value rather than monthly charges alone.

### Insight 5 — Use a combined risk profile
- **Claim:** No single variable fully describes churn risk.
- **Evidence:** The heatmap, contract segmentation, and engineered features highlight different aspects of customer behavior.
- **Business Action:** Combine contract flexibility, ticket velocity, tenure, and pricing into a simple retention-priority score.


In [ ]:
# TASK 7 — Submission Checklist
# 1. Run the notebook from the first cell to the last.
# 2. Download the completed .ipynb from Colab.
# 3. Add the notebook to the portfolio repository.
# 4. Commit and push the work, for example:
#    git add .
#    git commit -m "complete rewritten SaaS churn EDA"
#    git push origin main
# 5. Confirm the README includes findings, methodology, visualizations,
#    and recommended strategic actions.
